# PyTorch to LiteRT Conversion using Ai-edge-torch Repo

In [10]:
import ai_edge_torch
import numpy
import torch
import torchvision

from torch.ao.quantization.quantize_fx import convert_fx

from src.utils import load_data
from src.Quantization.utils.model_setup import setup_qat_student_model, quantization_mode
from src.utils import benchmark
from src.utils.model_setup import setup_model
from src.utils import test_inference, test_inference_onnx

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


### Conversion

In [26]:
device = torch.device("cpu")
pretrained_weights = f"models/SkinCancer/Quantized/quantized_student_state.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_qat_student_model(model_name="mobilenet_v2",num_classes=num_classes)

example_inputs = next(iter(dataloaders["train"]))[0].to(device)
student_model = quantization_mode(model, "fx", example_inputs=example_inputs)

# Move the model to CPU if needed (conversion is typically done on CPU).
student_model = student_model.to("cpu")

quantized_model = convert_fx(student_model)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
quantized_model.load_state_dict(state_dict)

# Set to eval mode.
quantized_model.eval()

Model prepared using FX Graph Mode QAT.


/home/jacob-delgado/anaconda3/envs/ECG_tf/lib/python3.12/site-packages/torch/ao/quantization/utils.py:408: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(


GraphModule(
  (features): Module(
    (0): Module(
      (0): QuantizedConvReLU2d(3, 32, kernel_size=(3, 3), stride=(2, 2), scale=0.23912420868873596, zero_point=0, padding=(1, 1))
    )
    (1): Module(
      (conv): Module(
        (0): Module(
          (0): QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.5721297860145569, zero_point=0, padding=(1, 1), groups=32)
        )
        (1): QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=1.2067992687225342, zero_point=148)
      )
    )
    (2): Module(
      (conv): Module(
        (0): Module(
          (0): QuantizedConvReLU2d(16, 96, kernel_size=(1, 1), stride=(1, 1), scale=0.9259501695632935, zero_point=0)
        )
        (1): Module(
          (0): QuantizedConvReLU2d(96, 96, kernel_size=(3, 3), stride=(2, 2), scale=0.10880327969789505, zero_point=0, padding=(1, 1), groups=96)
        )
        (2): QuantizedConv2d(96, 24, kernel_size=(1, 1), stride=(1, 1), scale=0.637142539024353, zero_po

In [27]:
print(type((example_inputs,)))
sample_inputs = (torch.randn(1, 3, 224, 224),)
print(type(sample_inputs))

<class 'tuple'>
<class 'tuple'>


In [30]:
edge_model = ai_edge_torch.convert(quantized_model, (example_inputs,))

# edge_output = edge_model(*example_inputs)

Unsupported: Failed running call_function <built-in method quantize_per_tensor of type object at 0x7eae69195f60>(*(FakeTensor(..., size=(32, 3, 224, 224)), FakeTensor(..., size=()), FakeTensor(..., size=(), dtype=torch.int64), torch.quint8), **{}):
quantized nyi in meta tensors

from user code:
   File "<eval_with_key>.11", line 7, in forward
    quantize_per_tensor = torch.quantize_per_tensor(x, features_0_0_input_scale_0, features_0_0_input_zero_point_0, torch.quint8);  x = features_0_0_input_scale_0 = features_0_0_input_zero_point_0 = None

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


In [4]:
if (numpy.allclose(
    torch_output.detach().numpy(),
    edge_output,
    atol=1e-5,
    rtol=1e-5,
)):
    print("Inference result with Pytorch and TfLite was within tolerance")
else:
    print("Something wrong with Pytorch --> TfLite")


Inference result with Pytorch and TfLite was within tolerance


In [5]:
edge_model.export('resnet.tflite')